<a href="https://colab.research.google.com/github/nds-najam/Chronos_Model_Comparison/blob/main/notebooks/03_chronos2_multivariate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Chronos-2 Multivariate Analysis



#### Imports and environment

In [22]:
!pip install uv
!uv pip install -q "chronos-forecasting==2.3.1"

In [23]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from chronos import Chronos2Pipeline

print("Python environment ready")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")

Python environment ready
PyTorch version: 2.11.0+cpu
CUDA available: False
Device: CPU


In [26]:
# Clone Github Repository
REPO_DIR = Path("/content/Chronos_Model_Comparison")
if REPO_DIR.exists():
  print("Repository exists:", REPO_DIR.exists())
  %cd /content/Chronos_Model_Comparison
else:
  !git clone https://github.com/nds-najam/Chronos_Model_Comparison.git

Repository exists: True
/content/Chronos_Model_Comparison


In [27]:
# Load ETTh1
#
DATA_URL = (
    "https://raw.githubusercontent.com/zhouhaoyi/"
    "ETDataset/main/ETT-small/ETTh1.csv"
)

df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

Shape: (17420, 8)
Columns:
['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


In [28]:
#
# Experiment Configuration
#
TARGET = "OT"

FEATURE_COLUMNS = [
    "HUFL",
    "HULL",
    "MUFL",
    "MULL",
    "LUFL",
    "LULL",
    "OT",
]

PREDICTION_LENGTH = 48

CONTEXT_LENGTHS = [
    512,
    1024,
    2048,
]

CROSS_LEARNING = False

print("Target:", TARGET)
print("Variables:", FEATURE_COLUMNS)
print("Prediction length:", PREDICTION_LENGTH)
print("Context lengths:", CONTEXT_LENGTHS)
print("Cross learning:", CROSS_LEARNING)

Target: OT
Variables: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
Prediction length: 48
Context lengths: [512, 1024, 2048]
Cross learning: False


In [29]:
#
# Prepare train/test data
#
values = df[FEATURE_COLUMNS].values.astype(np.float32)

history_end = len(values) - PREDICTION_LENGTH

full_history = values[:history_end]
future = values[history_end:]

print("Full historical data:", full_history.shape)
print("Future/test data:", future.shape)

Full historical data: (17372, 7)
Future/test data: (48, 7)


In [30]:
#
# Actual OT
#
OT_INDEX = FEATURE_COLUMNS.index(TARGET)

actual_ot = future[:, OT_INDEX]

print("Actual OT shape:", actual_ot.shape)
print("Actual OT range:", actual_ot.min(), actual_ot.max())

Actual OT shape: (48,)
Actual OT range: 8.371 12.381


In [31]:
#
# Load Chronos-2
#
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Chronos-2 on:", device)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
)

print("Chronos-2 loaded successfully")

Loading Chronos-2 on: cpu


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Chronos-2 loaded successfully


In [32]:
#
# MASE Function
#
def calculate_mase(
    actual: np.ndarray,
    forecast: np.ndarray,
    training_series: np.ndarray,
) -> float:
    """
    Calculate Mean Absolute Scaled Error using
    one-step naive forecast errors for scaling.
    """
    if len(training_series) < 2:
        raise ValueError("Training series must contain at least 2 observations.")

    naive_errors = np.abs(
        training_series[1:] - training_series[:-1]
    )

    scale = np.mean(naive_errors)

    if scale == 0:
        raise ValueError("MASE scaling factor is zero.")

    return float(
        np.mean(np.abs(actual - forecast)) / scale
    )

In [33]:
#
# Chronos-2 experiment function
#
def run_chronos2_experiment(
    context_length: int,
    cross_learning: bool = False,
) -> dict:
    """
    Run one Chronos-2 multivariate forecasting experiment.

    Parameters
    ----------
    context_length:
        Number of historical observations used by Chronos-2.

    cross_learning:
        Whether Chronos-2 cross-learning is enabled.

    Returns
    -------
    dict
        Forecast metrics and predictions.
    """

    if context_length > len(full_history):
        raise ValueError(
            f"context_length={context_length} exceeds "
            f"available history={len(full_history)}"
        )

    # Select the same final context window for every experiment.
    history = full_history[-context_length:]

    # Chronos-2 expects:
    # (n_series, n_variates, history_length)
    inputs = torch.tensor(
        history.T,
        dtype=torch.float32,
    ).unsqueeze(0)

    print(
        f"\nRunning C2 | "
        f"context={context_length} | "
        f"cross_learning={cross_learning}"
    )

    print("Input shape:", inputs.shape)

    start_time = time.perf_counter()

    quantiles, mean_forecast = pipeline.predict_quantiles(
        inputs,
        prediction_length=PREDICTION_LENGTH,
        context_length=context_length,
        cross_learning=cross_learning,
    )

    inference_time = time.perf_counter() - start_time

    # Extract OT forecast.
    forecast = (
        mean_forecast[0][OT_INDEX]
        .detach()
        .cpu()
        .numpy()
    )

    # Metrics.
    mae = mean_absolute_error(
        actual_ot,
        forecast,
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual_ot,
            forecast,
        )
    )

    training_ot = history[:, OT_INDEX]

    mase = calculate_mase(
        actual_ot,
        forecast,
        training_ot,
    )

    print(f"MASE: {mase:.6f}")
    print(f"MAE : {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"Time: {inference_time:.3f} sec")

    return {
        "model": "Chronos-2",
        "mode": "multivariate",
        "context_length": context_length,
        "cross_learning": cross_learning,
        "prediction_length": PREDICTION_LENGTH,
        "MASE": mase,
        "MAE": mae,
        "RMSE": rmse,
        "inference_time_seconds": inference_time,
        "forecast": forecast,
        "actual": actual_ot.copy(),
        "history": history[:, OT_INDEX].copy(),
    }